# 04. Agent DQN

Agent ghép các phần của DQN: chọn hành động bằng ε-greedy, tính Target theo Double DQN, cập nhật Q-Network và đồng bộ Target Network. Agent chỉ làm việc với mảng số và không import `gymnasium`.

Nội dung:

1. Lịch giảm ε.
2. Lớp `DQNAgent`.
3. Kiểm tra nhanh.

In [ ]:
%run ./02_q_network.ipynb

## 1. Lịch giảm ε

ε là xác suất chọn hành động ngẫu nhiên. ε giảm đều từ 1,0 (hoàn toàn ngẫu nhiên) về 0,01 trong 100.000 khung hình đầu, sau đó giữ nguyên.

In [ ]:
import numpy as np
import tensorflow as tf
import keras


def linear_epsilon(frame, start=1.0, end=0.01, decay_frames=100_000):
    fraction = min(frame / decay_frames, 1.0)
    return start + fraction * (end - start)

## 2. Lớp `DQNAgent`

Agent giữ hai mạng có cùng kiến trúc: mạng chính với trọng số w và Target Network với trọng số w⁻.

**Target theo Double DQN.** Mạng chính chọn hành động tốt nhất ở s', còn Target Network chấm điểm hành động đó:

```
a*     = argmax_a' Q(s', a'; w)
Target = r                          nếu terminated
Target = r + γ · Q(s', a*; w⁻)      trong mọi trường hợp còn lại, kể cả truncated
```

Hai mạng có sai số khác nhau nên ít khi cùng lệch lên, nhờ đó giảm việc đánh giá Q quá cao. Target được tính ngoài `GradientTape`, nên được coi là hằng số khi lấy gradient.

**Cập nhật.** Loss là MSE, tức trung bình của (Target − Q(s, a; w))², tối ưu bằng Adam với learning rate 0,0001.

**Tốc độ.** Hàm cập nhật và hàm chọn hành động được bọc trong `tf.function`. Không dùng `model.predict` để chọn hành động từng khung hình, vì nó chậm hơn khoảng 150 lần (mục 6 của `Implementation_Notes.md`).

**Theo dõi.** `update()` trả về loss và trung bình max Q của batch. Khi agent chơi tốt, Q của các trạng thái an toàn nên tiến về khoảng 11,9 đến 12,9. Q vượt rõ 12,9 là dấu hiệu đánh giá quá cao hoặc việc học đang phân kỳ.

**Lưu và tải.** Chỉ lưu trọng số mạng chính vào file `.weights.h5`. Khi tải, Target Network được chép lại từ mạng chính.

In [ ]:
class DQNAgent:
    def __init__(self, obs_dim=12, n_actions=2, gamma=0.99, learning_rate=1e-4,
                 hidden_units=256, seed=None):
        self.n_actions = n_actions
        self.gamma = gamma
        self.online = build_q_network(obs_dim, n_actions, hidden_units)
        self.target = build_q_network(obs_dim, n_actions, hidden_units)
        self.sync_target()
        self.optimizer = keras.optimizers.Adam(learning_rate)
        self._rng = np.random.default_rng(seed)

    # ---------- Chọn hành động ----------
    def act(self, state, epsilon):
        """ε-greedy: ngẫu nhiên với xác suất epsilon, ngược lại chọn Q lớn nhất."""
        if self._rng.random() < epsilon:
            return int(self._rng.integers(self.n_actions))
        return self.greedy_action(state)

    def greedy_action(self, state):
        state = np.asarray(state, dtype=np.float32)[None, :]
        return int(self._greedy(state).numpy()[0])

    @tf.function
    def _greedy(self, states):
        return tf.argmax(self.online(states, training=False), axis=1)

    def q_values(self, states):
        return self.online(np.asarray(states, dtype=np.float32), training=False).numpy()

    # ---------- Học ----------
    def update(self, states, actions, rewards, next_states, terminated):
        """Một bước Gradient Descent trên một batch. Trả về (loss, trung bình max Q)."""
        loss, mean_max_q = self._train_step(states, actions, rewards, next_states, terminated)
        return float(loss), float(mean_max_q)

    @tf.function
    def _train_step(self, states, actions, rewards, next_states, terminated):
        targets = self._double_dqn_targets(rewards, next_states, terminated)
        with tf.GradientTape() as tape:
            q_all = self.online(states, training=True)
            q_taken = tf.gather(q_all, actions, batch_dims=1)
            loss = tf.reduce_mean(tf.square(targets - q_taken))
        grads = tape.gradient(loss, self.online.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.online.trainable_variables))
        return loss, tf.reduce_mean(tf.reduce_max(q_all, axis=1))

    def _double_dqn_targets(self, rewards, next_states, terminated):
        best_actions = tf.argmax(self.online(next_states, training=False), axis=1)
        next_q = tf.gather(self.target(next_states, training=False), best_actions, batch_dims=1)
        return rewards + self.gamma * (1.0 - terminated) * next_q

    def sync_target(self):
        self.target.set_weights(self.online.get_weights())

    # ---------- Lưu và tải ----------
    def save(self, path):
        self.online.save_weights(path)

    def load(self, path):
        self.online.load_weights(path)
        self.sync_target()

## 3. Kiểm tra nhanh

Dùng dữ liệu giả để kiểm tra:

- Lịch ε đúng ở đầu, giữa và sau khi giảm xong.
- Target bằng đúng reward khi `terminated`, và bằng r + γ · Q(s', a*; w⁻) khi chưa.
- Cập nhật lặp lại trên cùng một batch làm loss giảm.
- Lưu rồi tải lại cho cùng giá trị Q.

In [ ]:
import os
import tempfile


def _self_check_agent():
    assert linear_epsilon(0) == 1.0
    assert abs(linear_epsilon(50_000) - 0.505) < 1e-9
    assert abs(linear_epsilon(200_000) - 0.01) < 1e-9

    rng = np.random.default_rng(0)
    batch = 32
    states = rng.uniform(-1, 1, size=(batch, 12)).astype(np.float32)
    actions = rng.integers(0, 2, size=batch).astype(np.int32)
    rewards = rng.choice([0.1, 1.0, -1.0], size=batch).astype(np.float32)
    next_states = rng.uniform(-1, 1, size=(batch, 12)).astype(np.float32)

    agent = DQNAgent(seed=0)
    agent.target.set_weights([w + 0.01 for w in agent.online.get_weights()])  # cho hai mạng khác nhau

    all_done = np.ones(batch, dtype=np.float32)
    assert np.allclose(agent._double_dqn_targets(rewards, next_states, all_done).numpy(), rewards)

    not_done = np.zeros(batch, dtype=np.float32)
    a_star = agent.q_values(next_states).argmax(axis=1)
    expected = rewards + 0.99 * agent.target(next_states).numpy()[np.arange(batch), a_star]
    got = agent._double_dqn_targets(rewards, next_states, not_done).numpy()
    assert np.allclose(got, expected, atol=1e-5)

    first_loss, _ = agent.update(states, actions, rewards, next_states, not_done)
    for _ in range(200):
        last_loss, _ = agent.update(states, actions, rewards, next_states, not_done)
    assert last_loss < first_loss, (first_loss, last_loss)

    assert agent.greedy_action(states[0]) in (0, 1)
    assert {agent.act(states[0], epsilon=1.0) for _ in range(50)} == {0, 1}

    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "agent.weights.h5")
        agent.save(path)
        restored = DQNAgent(seed=1)
        restored.load(path)
        assert np.allclose(restored.q_values(states), agent.q_values(states), atol=1e-6)
    print(f"04_agent: kiểm tra đạt. Loss {first_loss:.4f} -> {last_loss:.4f}")


_self_check_agent()